# E2 · The default RandomForestRegressor is not a random forest

**Type**: Find the bug · **Level**: ●●● challenge · **Time**: about 45 minutes · **extended path**

## Objective

Find out why a colleague gets the same RMSE (0.5216) with `RandomForestRegressor` and with `BaggingRegressor(DecisionTreeRegressor())` and concludes "the random forest adds nothing to bagging": in the regressor `max_features` is **1.0** (all the features), so there is no feature subsampling and the forest is a bagging. Fix it and measure the gain.

## Dataset

`ds.california()`, 75/25 split (`random_state=0`), RMSE with `squared=False`. Models with 100 trees, `random_state=0`, `n_jobs=-1`.

## Educational purpose

Recognize a tricky default: class names do not guarantee the algorithm. In the classifier `max_features="sqrt"`, in the regressor `1.0` (in recent versions; `"auto"` = all in older ones): for regression the default is a bagging of trees. It is always worth trying smaller values.

## Task

1. Run the colleague's cell: `rf_default_rmse` and `bagging_rmse`, and their difference.
2. Check the default: `default_max_features` (`RandomForestRegressor().max_features`) and `same_prediction` (True if the predictions of `rf_default` and `explicit_bagging` = `RandomForestRegressor(max_features=1.0)` match exactly on the test set).
3. Try `max_features` in `[0.2, 0.33, 0.5, 0.75, 1.0]`: save `mf_curve` (`pd.Series`, test RMSE) and `best_mf`.
4. Save `relative_gain = 1 − mf_curve[best_mf] / mf_curve[1.0]` and `prediction_corr` (correlation between the bagging predictions and those of the forest with `best_mf`: the trees are more different but the final predictions stay very similar).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.ensemble import BaggingRegressor, RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

X, y = ds.california()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)
rmse = lambda p: mean_squared_error(y_te, p, squared=False)

## Your work

### The colleague's code

In [ ]:
rf_default = RandomForestRegressor(100, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
bagging = BaggingRegressor(DecisionTreeRegressor(), n_estimators=100, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
rf_default_rmse, bagging_rmse = rmse(rf_default.predict(X_te)), rmse(bagging.predict(X_te))
print(f"RMSE: random forest {rf_default_rmse:.4f} | bagging {bagging_rmse:.4f} -> 'the random forest adds nothing to bagging'")

### Your diagnosis and the fix

In [ ]:
default_max_features = None
same_prediction = None

In [ ]:
mf_curve = pd.Series(dtype=float)
best_mf = None
relative_gain = None
prediction_corr = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`mf_curve = pd.Series({mf: rmse(RandomForestRegressor(100, max_features=mf, random_state=0, n_jobs=-1).fit(X_tr, y_tr).predict(X_te)) for mf in [0.2, 0.33, 0.5, 0.75, 1.0]})`. Correlation: `np.corrcoef(p_bagging, p_forest)[0, 1]`.'):
    check(abs(rf_default_rmse - bagging_rmse) < 0.002, "the colleague sees almost equal RMSEs (difference below 0.002)", "don't modify the colleague's code")
    check(default_max_features in (1.0, 1, "auto") and same_prediction is True, "the default is 1.0 (all the features) and the predictions match RandomForestRegressor(max_features=1.0)", "RandomForestRegressor().max_features")
    check(list(mf_curve.index) == [0.2, 0.33, 0.5, 0.75, 1.0] and best_mf in (0.33, 0.5), "the best is 0.33 or 0.5 (among the five values tried)", "idxmin of the curve")
    check(mf_curve[best_mf] < mf_curve[1.0] - 0.015 and relative_gain > 0.03, "the RMSE with the best value is over 0.015 below the default; relative gain over 3%", "1 - RMSE(best) / RMSE(1.0)")
    check(0.95 < prediction_corr < 1.0, "the predictions of the two forests are highly correlated (0.95-1.0) but not identical", "np.corrcoef of the predictions on the test set")
done()

---
The commented solution is in `solutions/E_find_the_bug/E2_the_default_regression_forest_is_a_bagging.ipynb`. Open it after you pass the check, or after you have really tried.